In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results"

EXPECTED_DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]

print("ROOT:", ROOT)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

ROOT: /home/stephane/resource-efficiency-benchmark
RESULTS_DIR exists: True


In [2]:
def discover_runs(results_dir: Path) -> pd.DataFrame:
    rows = []

    if not results_dir.exists():
        return pd.DataFrame(columns=["model", "run_id", "run_dir"])

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        model = model_dir.name

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            run_id = run_dir.name

            rows.append({
                "model": model,
                "run_id": run_id,
                "run_dir": str(run_dir),
            })

    return pd.DataFrame(rows)


def expected_files_for_run(run_dir: Path) -> dict:
    file_map = {
        "config_yaml": run_dir / "config.yaml",
        "server_log": run_dir / "server.log",
    }

    for ds in EXPECTED_DATASETS:
        file_map[f"{ds}_summary"] = run_dir / f"{ds}.csv"
        file_map[f"{ds}_samples"] = run_dir / f"{ds}_samples.csv"

    return file_map


def audit_run_files(runs_df: pd.DataFrame) -> pd.DataFrame:
    records = []

    for _, row in runs_df.iterrows():
        run_dir = Path(row["run_dir"])
        file_map = expected_files_for_run(run_dir)

        record = {
            "model": row["model"],
            "run_id": row["run_id"],
            "run_dir": row["run_dir"],
        }

        for key, path in file_map.items():
            record[key] = path.exists()

        records.append(record)

    return pd.DataFrame(records)


def list_present_dataset_files(run_dir: Path):
    summary_files = []
    sample_files = []

    for ds in EXPECTED_DATASETS:
        s = run_dir / f"{ds}.csv"
        sp = run_dir / f"{ds}_samples.csv"

        if s.exists():
            summary_files.append((ds, s))
        if sp.exists():
            sample_files.append((ds, sp))

    return summary_files, sample_files


def safe_read_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"FAILED reading {path}: {e}")
        return pd.DataFrame()


def collect_file_inventory(runs_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in runs_df.iterrows():
        run_dir = Path(row["run_dir"])
        summary_files, sample_files = list_present_dataset_files(run_dir)

        for dataset, path in summary_files:
            rows.append({
                "model": row["model"],
                "run_id": row["run_id"],
                "dataset": dataset,
                "file_type": "summary",
                "path": str(path),
            })

        for dataset, path in sample_files:
            rows.append({
                "model": row["model"],
                "run_id": row["run_id"],
                "dataset": dataset,
                "file_type": "samples",
                "path": str(path),
            })

    return pd.DataFrame(rows)


def schema_inventory(files_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in files_df.iterrows():
        path = Path(row["path"])
        df = safe_read_csv(path)

        rows.append({
            "model": row["model"],
            "run_id": row["run_id"],
            "dataset": row["dataset"],
            "file_type": row["file_type"],
            "path": row["path"],
            "n_rows": len(df),
            "n_cols": len(df.columns),
            "columns": list(df.columns),
        })

    return pd.DataFrame(rows)


def explode_columns(schema_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in schema_df.iterrows():
        for col in row["columns"]:
            rows.append({
                "model": row["model"],
                "run_id": row["run_id"],
                "dataset": row["dataset"],
                "file_type": row["file_type"],
                "column": col,
            })

    return pd.DataFrame(rows)

In [3]:
runs_df = discover_runs(RESULTS_DIR)
runs_df

,model,run_id,run_dir
0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...
1,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...
2,cpu,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...
3,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...
4,cpu,Phi-4-mini-instruct-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...
5,cpu,gemma-3-1b-it-Q8_0,/home/stephane/resource-efficiency-benchmark/r...
6,cpu,processed,/home/stephane/resource-efficiency-benchmark/r...
7,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...
8,pi5,Phi-4-mini-instruct-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...
9,pi5,SmolLM3-3B-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...


In [4]:
audit_df = audit_run_files(runs_df)
audit_df

,model,run_id,run_dir,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...,True,False,True,True,True,True,True,True,True,True
1,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...,True,False,True,True,True,True,True,True,True,True
2,cpu,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
3,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
4,cpu,Phi-4-mini-instruct-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
5,cpu,gemma-3-1b-it-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
6,cpu,processed,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
7,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...,True,False,True,True,True,True,True,True,True,True
8,pi5,Phi-4-mini-instruct-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
9,pi5,SmolLM3-3B-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False


In [5]:
if not audit_df.empty:
    display(
        audit_df.groupby("model").agg({
            "run_id": "nunique",
            "config_yaml": "sum",
            "server_log": "sum",
            "mmlu_summary": "sum",
            "mmlu_samples": "sum",
            "hellaswag_summary": "sum",
            "hellaswag_samples": "sum",
            "gsm8k_summary": "sum",
            "gsm8k_samples": "sum",
            "dolly_summary": "sum",
            "dolly_samples": "sum",
        }).rename(columns={"run_id": "n_runs"})
    )

,n_runs,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
model,,,,,,,,,,,
Ministral-3-3B-Instruct-2512-Q4_K_M,1,1,0,1,1,1,1,1,1,1,1
Phi-4-mini-instruct-Q4_K_M,1,1,0,1,1,1,1,1,1,1,1
cpu,5,0,0,0,0,0,0,0,0,0,0
gemma-3-1b-it-Q8_0,1,1,0,1,1,1,1,1,1,1,1
pi5,6,0,0,0,0,0,0,0,0,0,0
processed,6,0,0,0,0,0,0,0,0,0,0
qwen2,1,1,0,0,0,0,0,0,0,0,0
vllm,15,0,0,0,0,0,0,0,0,0,0


In [6]:
required_common = ["config_yaml", "server_log"]
dataset_file_cols = [
    "mmlu_summary", "mmlu_samples",
    "hellaswag_summary", "hellaswag_samples",
    "gsm8k_summary", "gsm8k_samples",
    "dolly_summary", "dolly_samples",
]

incomplete_df = audit_df[
    ~(audit_df[required_common].all(axis=1))
    | ~(audit_df[dataset_file_cols].any(axis=1))
]

incomplete_df

,model,run_id,run_dir,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...,True,False,True,True,True,True,True,True,True,True
1,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...,True,False,True,True,True,True,True,True,True,True
2,cpu,Llama-3.2-1B-Instruct-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
3,cpu,Ministral-3-3B-Instruct-2512-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
4,cpu,Phi-4-mini-instruct-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
5,cpu,gemma-3-1b-it-Q8_0,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
6,cpu,processed,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
7,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,/home/stephane/resource-efficiency-benchmark/r...,True,False,True,True,True,True,True,True,True,True
8,pi5,Phi-4-mini-instruct-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False
9,pi5,SmolLM3-3B-Q4_K_M,/home/stephane/resource-efficiency-benchmark/r...,False,False,False,False,False,False,False,False,False,False


In [7]:
files_df = collect_file_inventory(runs_df)
files_df.head(20)

,model,run_id,dataset,file_type,path
0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,summary,/home/stephane/resource-efficiency-benchmark/r...
1,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,summary,/home/stephane/resource-efficiency-benchmark/r...
2,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,summary,/home/stephane/resource-efficiency-benchmark/r...
3,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,summary,/home/stephane/resource-efficiency-benchmark/r...
4,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,samples,/home/stephane/resource-efficiency-benchmark/r...
5,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,samples,/home/stephane/resource-efficiency-benchmark/r...
6,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,samples,/home/stephane/resource-efficiency-benchmark/r...
7,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,samples,/home/stephane/resource-efficiency-benchmark/r...
8,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,mmlu,summary,/home/stephane/resource-efficiency-benchmark/r...
9,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,hellaswag,summary,/home/stephane/resource-efficiency-benchmark/r...


In [8]:
schema_df = schema_inventory(files_df)
schema_df[["model", "run_id", "dataset", "file_type", "n_rows", "n_cols", "path"]]

,model,run_id,dataset,file_type,n_rows,n_cols,path
0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,summary,1,24,/home/stephane/resource-efficiency-benchmark/r...
1,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,summary,1,24,/home/stephane/resource-efficiency-benchmark/r...
2,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,summary,1,24,/home/stephane/resource-efficiency-benchmark/r...
3,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,summary,1,22,/home/stephane/resource-efficiency-benchmark/r...
4,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,samples,1000,26,/home/stephane/resource-efficiency-benchmark/r...
5,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,samples,1000,26,/home/stephane/resource-efficiency-benchmark/r...
6,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,samples,1000,26,/home/stephane/resource-efficiency-benchmark/r...
7,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,samples,1000,26,/home/stephane/resource-efficiency-benchmark/r...
8,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,mmlu,summary,1,24,/home/stephane/resource-efficiency-benchmark/r...
9,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,hellaswag,summary,1,24,/home/stephane/resource-efficiency-benchmark/r...


In [9]:
summary_rows_check = schema_df[schema_df["file_type"] == "summary"][
    ["model", "run_id", "dataset", "n_rows", "path"]
].sort_values(["model", "run_id", "dataset"])

summary_rows_check

,model,run_id,dataset,n_rows,path
3,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,1,/home/stephane/resource-efficiency-benchmark/r...
2,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,1,/home/stephane/resource-efficiency-benchmark/r...
1,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,1,/home/stephane/resource-efficiency-benchmark/r...
0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,1,/home/stephane/resource-efficiency-benchmark/r...
11,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,dolly,1,/home/stephane/resource-efficiency-benchmark/r...
10,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,gsm8k,1,/home/stephane/resource-efficiency-benchmark/r...
9,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,hellaswag,1,/home/stephane/resource-efficiency-benchmark/r...
8,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,mmlu,1,/home/stephane/resource-efficiency-benchmark/r...
19,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,dolly,1,/home/stephane/resource-efficiency-benchmark/r...
18,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,gsm8k,1,/home/stephane/resource-efficiency-benchmark/r...


In [10]:
bad_summary_rows = summary_rows_check[summary_rows_check["n_rows"] != 1]
bad_summary_rows

,model,run_id,dataset,n_rows,path


In [11]:
sample_rows_check = schema_df[schema_df["file_type"] == "samples"][
    ["model", "run_id", "dataset", "n_rows", "path"]
].sort_values(["model", "run_id", "dataset"])

sample_rows_check

,model,run_id,dataset,n_rows,path
7,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,1000,/home/stephane/resource-efficiency-benchmark/r...
6,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,1000,/home/stephane/resource-efficiency-benchmark/r...
5,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,1000,/home/stephane/resource-efficiency-benchmark/r...
4,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,1000,/home/stephane/resource-efficiency-benchmark/r...
15,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,dolly,1000,/home/stephane/resource-efficiency-benchmark/r...
14,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,gsm8k,1000,/home/stephane/resource-efficiency-benchmark/r...
13,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,hellaswag,1000,/home/stephane/resource-efficiency-benchmark/r...
12,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,mmlu,1000,/home/stephane/resource-efficiency-benchmark/r...
23,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,dolly,1000,/home/stephane/resource-efficiency-benchmark/r...
22,gemma-3-1b-it-Q8_0,2026-07-10_13-56-53,gsm8k,1000,/home/stephane/resource-efficiency-benchmark/r...


In [12]:
columns_df = explode_columns(schema_df)

columns_by_group = (
    columns_df.groupby(["dataset", "file_type"])["column"]
    .unique()
    .reset_index()
)

columns_by_group

,dataset,file_type,column
0,dolly,samples,"[model_name, served_model_id, model_path, data..."
1,dolly,summary,"[Model name, Model path, Dolly Served model id..."
2,gsm8k,samples,"[model_name, served_model_id, model_path, data..."
3,gsm8k,summary,"[Model name, Model path, Served model id, All ..."
4,hellaswag,samples,"[model_name, served_model_id, model_path, data..."
5,hellaswag,summary,"[Model name, Model path, Served model id, All ..."
6,mmlu,samples,"[model_name, served_model_id, model_path, data..."
7,mmlu,summary,"[Model name, Model path, Served model id, All ..."


In [13]:
schema_signature_df = schema_df.copy()
schema_signature_df["schema_signature"] = schema_signature_df["columns"].apply(lambda cols: tuple(cols))

schema_consistency = (
    schema_signature_df.groupby(["dataset", "file_type"])["schema_signature"]
    .nunique()
    .reset_index(name="n_distinct_schemas")
)

schema_consistency

,dataset,file_type,n_distinct_schemas
0,dolly,samples,1
1,dolly,summary,1
2,gsm8k,samples,1
3,gsm8k,summary,1
4,hellaswag,samples,1
5,hellaswag,summary,1
6,mmlu,samples,1
7,mmlu,summary,1


inconsistent_schemas = schema_consistency[schema_consistency["n_distinct_schemas"] > 1]
inconsistent_schemas

In [14]:
for dataset in EXPECTED_DATASETS:
    for file_type in ["summary", "samples"]:
        subset = files_df[(files_df["dataset"] == dataset) & (files_df["file_type"] == file_type)]
        if subset.empty:
            continue

        path = Path(subset.iloc[0]["path"])
        df = safe_read_csv(path)

        print(f"\n=== {dataset} / {file_type} ===")
        print(path)
        display(df.head(3))


=== mmlu / summary ===
/home/stephane/resource-efficiency-benchmark/results/Ministral-3-3B-Instruct-2512-Q4_K_M/2026-07-10_13-56-53/mmlu.csv


,Model name,Model path,Served model id,All served model ids,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,0.64,0.0,4893.4554,39.5872,5014.0043,40.6731,8.433,0.144,56.6541,0.1622,66.5763,0.4863,0.1977,0.0016,0.1977,0.0016,94.4,10818.85,4812.91,81.0



=== mmlu / samples ===
/home/stephane/resource-efficiency-benchmark/results/Ministral-3-3B-Instruct-2512-Q4_K_M/2026-07-10_13-56-53/mmlu_samples.csv


,model_name,served_model_id,model_path,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,system_metrics_trace_json
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,1,A disk of radius 0.1 m initially at rest under...,A disk of radius 0.1 m initially at rest under...,0.4 m || 0.8 m || 1.2 m || 1.6 m,NaN,NaN,3,C,False,C,1,2540.8694,2620.0589,12.6279,54.0667,88.2,10409.49,4425.21,66.6667,69.0,"[{""cpu_percent"":1.2,""ram_used_mb"":10403.47,""ra..."
1,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,2,"Cultural homogenization.\n\nA. 1,3,4\nB. 1,2,3...",Cultural homogenization.,"1,3,4 || 1,2,3 || 1,2,3,4 || 2,3,4",NaN,NaN,1,C,False,C,1,1559.0006,1642.8462,11.9267,52.0000,84.8,10411.58,4425.22,70.3333,71.0,"[{""cpu_percent"":2.6,""ram_used_mb"":10409.49,""ra..."
2,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,3,Monte Alban is located in the ________ and was...,Monte Alban is located in the ________ and was...,Valley of Mexico; Aztec. || Valley of Oaxaca; ...,NaN,NaN,1,B,True,B,1,1840.5358,1920.8730,12.4475,53.1333,85.7,10414.09,4425.23,72.3333,73.0,"[{""cpu_percent"":2.4,""ram_used_mb"":10411.82,""ra..."



=== hellaswag / summary ===
/home/stephane/resource-efficiency-benchmark/results/Ministral-3-3B-Instruct-2512-Q4_K_M/2026-07-10_13-56-53/hellaswag.csv


...

,model_name,served_model_id,model_path,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,system_metrics_trace_json
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,gsm8k,1,1,A news website publishes an average of 20 poli...,A news website publishes an average of 20 poli...,NaN,NaN,NaN,840,840.0,True,"In February with **28 days**, here’s the calcu...",55,2489.7062,15659.1979,4.1763,76.3937,89.6,10529.90,4425.49,62.1746,63.0,"[{""cpu_percent"":1.6,""ram_used_mb"":10529.9,""ram..."
1,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,gsm8k,1,2,Joe's bag of Halloween candy has 25 chocolate ...,Joe's bag of Halloween candy has 25 chocolate ...,NaN,NaN,NaN,2600,1600.0,False,Let's break down the problem step by step to f...,148,2524.8907,30655.5317,5.2612,76.7215,89.9,10544.89,4425.53,62.3385,63.0,"[{""cpu_percent"":4.3,""ram_used_mb"":10527.05,""ra..."
2,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,gsm8k,1,3,Alisa biked 12 miles per hour for 4.5 hours. S...,Alisa biked 12 miles per hour for 4.5 hours. S...,NaN,NaN,NaN,79,79.0,True,To find the total miles biked by Alisa and Sta...,107,2075.3689,27436.7454,4.2190,76.6615,87.8,10539.06,4425.55,62.3333,63.0,"[{""cpu_percent"":3.8,""ram_used_mb"":10520.07,""ra..."



=== dolly / summary ===
/home/stephane/resource-efficiency-benchmark/results/Ministral-3-3B-Instruct-2512-Q4_K_M/2026-07-10_13-56-53/dolly.csv


,Model name,Model path,Dolly Served model id,Dolly All served model ids,Dolly TTFT ms mean,Dolly TTFT ms sd,Dolly End-to-end latency ms mean,Dolly End-to-end latency ms sd,Dolly Tokens/sec mean,Dolly Tokens/sec sd,Dolly CPU percent avg mean,Dolly CPU percent avg sd,Dolly Temp C avg mean,Dolly Temp C avg sd,Dolly Sequential sustained tokens/sec mean,Dolly Sequential sustained tokens/sec sd,Dolly Sequential sustained samples/sec mean,Dolly Sequential sustained samples/sec sd,Dolly CPU percent peak,Dolly RAM used MB peak,Dolly Process RAM MB peak,Dolly Temp C peak
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,4023.8547,635.0233,24905.1323,1126.6742,5.7768,0.1101,76.6755,0.208,65.9418,0.7396,4.8522,0.1632,0.0402,0.0018,94.1,12296.67,6207.45,81.0



=== dolly / samples ===
/home/stephane/resource-efficiency-benchmark/results/Ministral-3-3B-Instruct-2512-Q4_K_M/2026-07-10_13-56-53/dolly_samples.csv


,model_name,served_model_id,model_path,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,system_metrics_trace_json
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,dolly,1,1,Write some important points about the National...,NaN,NaN,Write some important points about the National...,The National Basketball League (NBL) was a pro...,NaN,NaN,NaN,Here are the key important points about the **...,102,4191.1225,23246.5771,5.3528,76.744,90.9,10674.82,4542.45,66.3077,67.0,"[{""cpu_percent"":1.1,""ram_used_mb"":10662.86,""ra..."
1,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,dolly,1,2,Why do people enjoy baseball?\n\nResponse:,NaN,NaN,Why do people enjoy baseball?,NaN,NaN,NaN,NaN,Baseball is enjoyed by millions of people arou...,167,487.7332,26711.6293,6.3682,76.859,81.6,10680.89,4542.48,66.9180,68.0,"[{""cpu_percent"":2.0,""ram_used_mb"":10645.8,""ram..."
2,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,dolly,1,3,Plan a 10-15 day trip from bay area to Arizona...,NaN,NaN,Plan a 10-15 day trip from bay area to Arizona...,NaN,NaN,NaN,NaN,Here’s a well-balanced **10–14 day itinerary**...,133,1215.9310,27562.2385,5.0481,76.208,85.7,10664.89,4542.50,66.8880,68.0,"[{""cpu_percent"":5.6,""ram_used_mb"":10650.46,""ra..."


In [15]:
summary_tables = []

for _, row in files_df[files_df["file_type"] == "summary"].iterrows():
    path = Path(row["path"])
    df = safe_read_csv(path)

    if df.empty:
        continue

    df["model_dir"] = row["model"]
    df["run_id"] = row["run_id"]
    df["dataset"] = row["dataset"]
    df["source_path"] = str(path)

    summary_tables.append(df)

all_summaries_df = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
all_summaries_df.head()

,Model name,Model path,Served model id,All served model ids,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,model_dir,run_id,dataset,source_path,Dolly Served model id,Dolly All served model ids,Dolly TTFT ms mean,Dolly TTFT ms sd,Dolly End-to-end latency ms mean,Dolly End-to-end latency ms sd,Dolly Tokens/sec mean,Dolly Tokens/sec sd,Dolly CPU percent avg mean,Dolly CPU percent avg sd,Dolly Temp C avg mean,Dolly Temp C avg sd,Dolly Sequential sustained tokens/sec mean,Dolly Sequential sustained tokens/sec sd,Dolly Sequential sustained samples/sec mean,Dolly Sequential sustained samples/sec sd,Dolly CPU percent peak,Dolly RAM used MB peak,Dolly Process RAM MB peak,Dolly Temp C peak
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,0.640,0.0000,4893.4554,39.5872,5014.0043,40.6731,8.4330,0.1440,56.6541,0.1622,66.5763,0.4863,0.1977,0.0016,0.1977,0.0016,94.4,10818.85,4812.91,81.0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,0.420,0.0000,6620.0853,39.4345,6741.4896,40.3805,8.3674,0.1284,56.9972,0.0981,66.6310,0.1852,0.1474,0.0009,0.1474,0.0009,94.5,10717.30,4426.10,81.0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,hellaswag,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,0.519,0.0032,2482.0997,41.9336,24602.6782,360.3690,5.3746,0.0829,77.2852,0.3979,64.5473,0.7946,4.8328,0.0718,0.0406,0.0006,92.0,11042.84,4434.47,80.0,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,gsm8k,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,dolly,/home/stephane/resource-efficiency-benchmark/r...,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf | Min...,4023.8547,635.0233,24905.1323,1126.6742,5.7768,0.1101,76.6755,0.208,65.9418,0.7396,4.8522,0.1632,0.0402,0.0018,94.1,12296.67,6207.45,81.0
4,Phi-4-mini-instruct-Q4_K_M.gguf,/app/models/Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf,Phi-4-mini-instruct-Q4_K_M.gguf | Phi-4-mini-i...,0.540,0.0000,4809.2242,47.4001,5211.8453,52.6795,6.5144,0.1341,56.9146,0.2706,69.6363,0.9511,0.4790,0.0045,0.1904,0.0019,95.4,12329.49,6683.48,83.0,Phi-4-mini-instruct-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
sample_tables = []

for _, row in files_df[files_df["file_type"] == "samples"].iterrows():
    path = Path(row["path"])
    df = safe_read_csv(path)

    if df.empty:
        continue

    df["model_dir"] = row["model"]
    df["run_id"] = row["run_id"]
    df["dataset"] = row["dataset"]
    df["source_path"] = str(path)

    sample_tables.append(df)

all_samples_df = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()
all_samples_df.head()

,model_name,served_model_id,model_path,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,system_metrics_trace_json,model_dir,run_id,dataset,source_path
0,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,1,A disk of radius 0.1 m initially at rest under...,A disk of radius 0.1 m initially at rest under...,0.4 m || 0.8 m || 1.2 m || 1.6 m,NaN,NaN,3.0,C,False,C,1,2540.8694,2620.0589,12.6279,54.0667,88.2,10409.49,4425.21,66.6667,69.0,"[{""cpu_percent"":1.2,""ram_used_mb"":10403.47,""ra...",Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,2,"Cultural homogenization.\n\nA. 1,3,4\nB. 1,2,3...",Cultural homogenization.,"1,3,4 || 1,2,3 || 1,2,3,4 || 2,3,4",NaN,NaN,1.0,C,False,C,1,1559.0006,1642.8462,11.9267,52.0000,84.8,10411.58,4425.22,70.3333,71.0,"[{""cpu_percent"":2.6,""ram_used_mb"":10409.49,""ra...",Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...
2,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,3,Monte Alban is located in the ________ and was...,Monte Alban is located in the ________ and was...,Valley of Mexico; Aztec. || Valley of Oaxaca; ...,NaN,NaN,1.0,B,True,B,1,1840.5358,1920.8730,12.4475,53.1333,85.7,10414.09,4425.23,72.3333,73.0,"[{""cpu_percent"":2.4,""ram_used_mb"":10411.82,""ra...",Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...
3,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,4,During a clinical study examining the effects ...,During a clinical study examining the effects ...,Pulse pressure || Stroke volume || Systolic bl...,NaN,NaN,3.0,C,False,C,1,2975.8564,3071.1838,10.4902,52.0333,88.3,10414.09,4425.23,73.0000,73.0,"[{""cpu_percent"":4.5,""ram_used_mb"":10414.09,""ra...",Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...
4,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,Ministral-3-3B-Instruct-2512-Q4_K_M.gguf,/app/models/Ministral-3-3B-Instruct-2512-Q4_K_...,mmlu,1,5,A 17-year-old girl is brought to the physician...,A 17-year-old girl is brought to the physician...,Hyperthyroidism || Hypogonadotropic hypogonadi...,NaN,NaN,1.0,B,True,B,1,4822.1397,4902.7506,12.4053,55.7000,89.3,10414.78,4425.25,75.6667,77.0,"[{""cpu_percent"":4.5,""ram_used_mb"":10406.84,""ra...",Ministral-3-3B-Instruct-2512-Q4_K_M,2026-07-10_13-56-53,mmlu,/home/stephane/resource-efficiency-benchmark/r...


In [17]:
if not all_summaries_df.empty:
    print("Summary shape:", all_summaries_df.shape)
    print("\nDatasets in summaries:", sorted(all_summaries_df["dataset"].dropna().unique()))
    print("\nRuns per model:")
    display(all_summaries_df.groupby("model_dir")["run_id"].nunique().rename("n_runs").reset_index())

    print("\nNull counts:")
    display(all_summaries_df.isna().sum().sort_values(ascending=False).head(30))

Summary shape: (12, 48)

Datasets in summaries: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']

Runs per model:


,model_dir,n_runs
0,Ministral-3-3B-Instruct-2512-Q4_K_M,1
1,Phi-4-mini-instruct-Q4_K_M,1
2,gemma-3-1b-it-Q8_0,1



Null counts:


Dolly End-to-end latency ms mean               9
Dolly End-to-end latency ms sd                 9
Dolly Tokens/sec mean                          9
Dolly Tokens/sec sd                            9
Dolly CPU percent avg mean                     9
Dolly CPU percent avg sd                       9
Dolly Temp C avg mean                          9
Dolly Temp C avg sd                            9
Dolly Sequential sustained tokens/sec mean     9
Dolly Sequential sustained tokens/sec sd       9
Dolly Sequential sustained samples/sec mean    9
Dolly Sequential sustained samples/sec sd      9
Dolly TTFT ms sd                               9
Dolly TTFT ms mean                             9
Dolly All served model ids                     9
Dolly Served model id                          9
Dolly CPU percent peak                         9
Dolly RAM used MB peak                         9
Dolly Process RAM MB peak                      9
Dolly Temp C peak                              9
Tokens/sec sd       

In [18]:
if not all_samples_df.empty:
    print("Samples shape:", all_samples_df.shape)
    print("\nDatasets in samples:", sorted(all_samples_df["dataset"].dropna().unique()))
    print("\nRows by model/dataset:")
    display(
        all_samples_df.groupby(["model_dir", "dataset"])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values(["model_dir", "dataset"])
    )

    print("\nNull counts:")
    display(all_samples_df.isna().sum().sort_values(ascending=False).head(30))

Samples shape: (12000, 30)

Datasets in samples: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']

Rows by model/dataset:


,model_dir,dataset,n_rows
0,Ministral-3-3B-Instruct-2512-Q4_K_M,dolly,1000
1,Ministral-3-3B-Instruct-2512-Q4_K_M,gsm8k,1000
2,Ministral-3-3B-Instruct-2512-Q4_K_M,hellaswag,1000
3,Ministral-3-3B-Instruct-2512-Q4_K_M,mmlu,1000
4,Phi-4-mini-instruct-Q4_K_M,dolly,1000
5,Phi-4-mini-instruct-Q4_K_M,gsm8k,1000
6,Phi-4-mini-instruct-Q4_K_M,hellaswag,1000
7,Phi-4-mini-instruct-Q4_K_M,mmlu,1000
8,gemma-3-1b-it-Q8_0,dolly,1000
9,gemma-3-1b-it-Q8_0,gsm8k,1000



Null counts:


context                      11115
instruction                   9000
choices                       6000
question                      3000
gold_answer                   3000
parsed_prediction             3000
correct                       3000
run_idx                          0
sample_idx                       0
model_path                       0
served_model_id                  0
model_name                       0
dataset_name                     0
prompt                           0
raw_output                       0
output_tokens                    0
ttft_ms                          0
end_to_end_latency_ms            0
tokens_per_sec                   0
cpu_percent_avg                  0
cpu_percent_peak                 0
ram_used_mb_peak                 0
process_ram_mb_peak              0
temp_c_avg                       0
temp_c_peak                      0
system_metrics_trace_json        0
model_dir                        0
run_id                           0
dataset             

In [19]:
if not all_samples_df.empty:
    numeric_checks = {}

    for col in [
        "ttft_ms",
        "end_to_end_latency_ms",
        "tokens_per_sec",
        "output_tokens",
        "cpu_percent_avg",
        "cpu_percent_peak",
        "ram_used_mb_peak",
        "process_ram_mb_peak",
        "temp_c_avg",
        "temp_c_peak",
    ]:
        if col in all_samples_df.columns:
            numeric_checks[col] = {
                "min": pd.to_numeric(all_samples_df[col], errors="coerce").min(),
                "max": pd.to_numeric(all_samples_df[col], errors="coerce").max(),
                "n_negative": (pd.to_numeric(all_samples_df[col], errors="coerce") < 0).sum(),
            }

    numeric_checks_df = pd.DataFrame(numeric_checks).T
    numeric_checks_df

In [20]:
if not all_samples_df.empty and "correct" in all_samples_df.columns:
    correctness_by_dataset = (
        all_samples_df.groupby("dataset")["correct"]
        .agg(["count", "nunique"])
        .reset_index()
    )
    correctness_by_dataset

In [21]:
if not all_samples_df.empty and "parsed_prediction" in all_samples_df.columns:
    display(
        all_samples_df.groupby("dataset")["parsed_prediction"]
        .apply(lambda s: s.notna().mean())
        .rename("parsed_prediction_non_null_rate")
        .reset_index()
    )

,dataset,parsed_prediction_non_null_rate
0,dolly,0.0
1,gsm8k,1.0
2,hellaswag,1.0
3,mmlu,1.0


In [22]:
if not all_samples_df.empty and {"model_dir", "run_id", "dataset", "run_idx"}.issubset(all_samples_df.columns):
    per_measured_run_counts = (
        all_samples_df.groupby(["model_dir", "run_id", "dataset", "run_idx"])
        .size()
        .rename("n_samples")
        .reset_index()
        .sort_values(["model_dir", "run_id", "dataset", "run_idx"])
    )
    per_measured_run_counts.head(50)

In [23]:
if not all_samples_df.empty and {"model_dir", "run_id", "dataset", "run_idx"}.issubset(all_samples_df.columns):
    count_consistency = (
        per_measured_run_counts.groupby(["model_dir", "run_id", "dataset"])["n_samples"]
        .agg(["min", "max", "nunique"])
        .reset_index()
        .sort_values(["model_dir", "run_id", "dataset"])
    )
    count_consistency

In [24]:
if not all_samples_df.empty and "run_idx" in all_samples_df.columns:
    display(
        all_samples_df.groupby("dataset")["run_idx"]
        .agg(["min", "max", "nunique"])
        .reset_index()
    )

,dataset,min,max,nunique
0,dolly,1,10,10
1,gsm8k,1,10,10
2,hellaswag,1,10,10
3,mmlu,1,10,10


In [25]:
dolly_summary = all_summaries_df[all_summaries_df["dataset"] == "dolly"].copy()
if not dolly_summary.empty:
    print("Dolly columns:")
    for c in dolly_summary.columns:
        print(c)

Dolly columns:
Model name
Model path
Served model id
All served model ids
Accuracy mean
Accuracy sd
TTFT ms mean
TTFT ms sd
End-to-end latency ms mean
End-to-end latency ms sd
Tokens/sec mean
Tokens/sec sd
CPU percent avg mean
CPU percent avg sd
Temp C avg mean
Temp C avg sd
Sequential sustained tokens/sec mean
Sequential sustained tokens/sec sd
Sequential sustained samples/sec mean
Sequential sustained samples/sec sd
CPU percent peak
RAM used MB peak
Process RAM MB peak
Temp C peak
model_dir
run_id
dataset
source_path
Dolly Served model id
Dolly All served model ids
Dolly TTFT ms mean
Dolly TTFT ms sd
Dolly End-to-end latency ms mean
Dolly End-to-end latency ms sd
Dolly Tokens/sec mean
Dolly Tokens/sec sd
Dolly CPU percent avg mean
Dolly CPU percent avg sd
Dolly Temp C avg mean
Dolly Temp C avg sd
Dolly Sequential sustained tokens/sec mean
Dolly Sequential sustained tokens/sec sd
Dolly Sequential sustained samples/sec mean
Dolly Sequential sustained samples/sec sd
Dolly CPU percent pe

Observations to capture here:
- which models/runs are complete
- whether summary CSVs are consistently one-row files
- whether sample CSV counts are stable across measured runs
- whether schemas are consistent across runs
- whether Dolly summary needs special normalization
- any obvious null or metric anomalies